In [ ]:
# In[1]:

import pandas as pd
import pytz
from pathlib import Path

tz = pytz.timezone('Asia/Shanghai')

# Helper to load and validate a CSV file with expected columns
def load_and_validate(path, expected_cols):
    p = Path(path)
    if not p.exists():
        return None, f"INVALID_FILE: {path} does not exist"
    df = pd.read_csv(p)
    if df.empty:
        return None, f"EMPTY_FILE: {path} is empty"
    missing = [c for c in expected_cols if c not in df.columns]
    if missing:
        return None, f"MISSING_COLUMNS: {path} missing columns {missing}"
    return df, None

# Load metric.csv
metric_df, metric_err = load_and_validate("metric.csv", ["timestamp", "cmdb_id", "kpi_name", "value"])
if metric_err:
    metric_overall = metric_err
    metric_top50 = pd.DataFrame()  # empty placeholder
else:
    # Ensure numeric value
    metric_df["value"] = pd.to_numeric(metric_df["value"], errors="coerce")
    # overall min/max timestamps (as int and as Asia/Shanghai datetime)
    ts_min = int(metric_df["timestamp"].min())
    ts_max = int(metric_df["timestamp"].max())
    ts_min_dt = pd.to_datetime(ts_min, unit="s", utc=True).tz_convert(tz)
    ts_max_dt = pd.to_datetime(ts_max, unit="s", utc=True).tz_convert(tz)
    metric_overall = pd.DataFrame([{
        "min_timestamp": ts_min,
        "min_timestamp_asia_shanghai": ts_min_dt,
        "max_timestamp": ts_max,
        "max_timestamp_asia_shanghai": ts_max_dt,
        "rows": len(metric_df)
    }])
    # group by (cmdb_id, kpi_name): count_of_points (non-null value) and global P95 of value
    grp = metric_df.groupby(["cmdb_id", "kpi_name"], as_index=False).agg(
        count_of_points = ("value", "count"),
        p95_value = ("value", lambda s: float(s.quantile(0.95)) if s.notna().any() else float("nan"))
    )
    metric_top50 = grp.sort_values("count_of_points", ascending=False).head(50)[["cmdb_id","kpi_name","count_of_points","p95_value"]]

# Load trace.csv
trace_df, trace_err = load_and_validate("trace.csv", ["timestamp", "cmdb_id", "trace_name", "value"])
if trace_err:
    trace_overall = trace_err
    trace_top50 = pd.DataFrame()
else:
    trace_df["value"] = pd.to_numeric(trace_df["value"], errors="coerce")
    ts_min = int(trace_df["timestamp"].min())
    ts_max = int(trace_df["timestamp"].max())
    ts_min_dt = pd.to_datetime(ts_min, unit="s", utc=True).tz_convert(tz)
    ts_max_dt = pd.to_datetime(ts_max, unit="s", utc=True).tz_convert(tz)
    trace_overall = pd.DataFrame([{
        "min_timestamp": ts_min,
        "min_timestamp_asia_shanghai": ts_min_dt,
        "max_timestamp": ts_max,
        "max_timestamp_asia_shanghai": ts_max_dt,
        "rows": len(trace_df)
    }])
    grp_t = trace_df.groupby(["cmdb_id", "trace_name"], as_index=False).agg(
        count_of_points = ("value", "count"),
        p95_value = ("value", lambda s: float(s.quantile(0.95)) if s.notna().any() else float("nan"))
    )
    trace_top50 = grp_t.sort_values("count_of_points", ascending=False).head(50)[["cmdb_id","trace_name","count_of_points","p95_value"]]

# Display compact results (variables)
metric_overall, metric_top50, trace_overall, trace_top50

```
Out[1]:
```
```python
# Summarize the previously computed aggregation results into a compact plain-English summary string.
# Reuse existing variables: metric_overall, metric_top50, trace_overall, trace_top50

def fmt_ts(x):
    try:
        return x.isoformat()
    except Exception:
        return str(x)

lines = []
# Metric summary
if isinstance(metric_overall, str):
    lines.append(f"METRIC CSV: {metric_overall}")
else:
    row = metric_overall.iloc[0]
    lines.append("METRIC CSV:")
    lines.append(f" - Time range (epoch sec): {int(row['min_timestamp'])} to {int(row['max_timestamp'])}")
    lines.append(f" - Time range (Asia/Shanghai): {fmt_ts(row['min_timestamp_asia_shanghai'])} to {fmt_ts(row['max_timestamp_asia_shanghai'])}")
    lines.append(f" - Total metric rows: {int(row['rows'])}")
    if isinstance(metric_top50, str) or metric_top50 is None or metric_top50.shape[0] == 0:
        lines.append(" - No per-(cmdb_id,kpi_name) aggregation available.")
    else:
        lines.append(" - Top metric (cmdb_id, kpi_name) by count (top 5):")
        for i, r in metric_top50.head(5).reset_index(drop=True).iterrows():
            lines.append(f"    {i+1}. {r['cmdb_id']} | {r['kpi_name']} — count: {int(r['count_of_points'])}, p95: {r['p95_value']:.6g}")

# Trace summary
if isinstance(trace_overall, str):
    lines.append(f"TRACE CSV: {trace_overall}")
else:
    row = trace_overall.iloc[0]
    lines.append("TRACE CSV:")
    lines.append(f" - Time range (epoch sec): {int(row['min_timestamp'])} to {int(row['max_timestamp'])}")
    lines.append(f" - Time range (Asia/Shanghai): {fmt_ts(row['min_timestamp_asia_shanghai'])} to {fmt_ts(row['max_timestamp_asia_shanghai'])}")
    lines.append(f" - Total trace rows: {int(row['rows'])}")
    if isinstance(trace_top50, str) or trace_top50 is None or trace_top50.shape[0] == 0:
        lines.append(" - No per-(cmdb_id,trace_name) aggregation available.")
    else:
        lines.append(" - Top trace (cmdb_id, trace_name) by count (top 5):")
        for i, r in trace_top50.head(5).reset_index(drop=True).iterrows():
            lines.append(f"    {i+1}. {r['cmdb_id']} | {r['trace_name']} — count: {int(r['count_of_points'])}, p95: {r['p95_value']:.6g}")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   min_timestamp min_timestamp_asia_shanghai  max_timestamp max_timestamp_asia_shanghai   rows
0     1590508800   2020-05-27 00:00:00+08:00     1590512400   2020-05-27 01:00:00+08:00  89993,      cmdb_id                  kpi_name  count_of_points    p95_value
97    db_003              CPU_free_pct               61    98.933695
987   os_007               Disk_rd_ios               61     0.000000
1527  os_017                Send_total               61     0.259472
5     db_001              CPU_free_pct               61    99.260050
1772  os_022  Outgoing_network_traffic               61     0.068280
...      ...                       ...              ...          ...
1513  os_017               Memory_used               60     5.734058
812   os_003                Recv_total               60  2547.318675
813   os_003                Send_total               60     3.757925
1216  os_011      Processor_load_5_min               60     0.016834
1220  os_011                Recv_total               60    15.628013

[50 rows x 4 columns],    min_timestamp min_timestamp_asia_shanghai  max_timestamp max_timestamp_asia_shanghai  rows
0     1590510600   2020-05-27 00:30:00+08:00     1590512340   2020-05-27 00:59:00+08:00  6340,        cmdb_id                                       trace_name  count_of_points    p95_value
0   docker_001    trace.from_os_022.remoteprocess.duration_mean               30    35.730468
1   docker_001     trace.from_os_022.remoteprocess.duration_p95               30    45.132622
2   docker_001       trace.from_os_022.remoteprocess.error_rate               30     0.000000
3   docker_001        trace.from_os_022.remoteprocess.row_count               30   133.550000
4   docker_001                     trace.self.csf.duration_mean               30     2.666777
..         ...                                              ...              ...          ...
45  docker_002                    trace.self.local.duration_p95               30     0.091550
46  docker_002                      trace.self.local.error_rate               30     0.000000
47  docker_002                       trace.self.local.row_count               30  1929.900000
48  docker_002  trace.to_docker_005.remoteprocess.duration_mean               30     3.571470
49  docker_002   trace.to_docker_005.remoteprocess.duration_p95               30     4.252685

[50 rows x 4 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
from pathlib import Path

tz = pytz.timezone('Asia/Shanghai')
UTC = pytz.UTC

# Helper to (re)load files if needed
def load_csv_if_missing(varname, filename, expected_cols):
    if varname in globals() and globals()[varname] is not None:
        return globals()[varname]
    p = Path(filename)
    if not p.exists():
        raise FileNotFoundError(f"{filename} not found")
    df = pd.read_csv(p)
    missing = [c for c in expected_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{filename} missing columns {missing}")
    return df

# Load metric and trace dataframes if not present
metric_df = load_csv_if_missing('metric_df', 'metric.csv', ['timestamp','cmdb_id','kpi_name','value'])
trace_df = load_csv_if_missing('trace_df', 'trace.csv', ['timestamp','cmdb_id','trace_name','value'])

# Ensure numeric values
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Ensure metric_top50 and trace_top50 exist (compute if missing) using full series for global P95
if 'metric_top50' not in globals() or metric_top50 is None or metric_top50.shape[0]==0:
    grp = metric_df.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
        count_of_points = ('value','count'),
        p95_value = ('value', lambda s: float(s.quantile(0.95)) if s.notna().any() else float('nan'))
    )
    metric_top50 = grp.sort_values('count_of_points', ascending=False).head(50)[['cmdb_id','kpi_name','count_of_points','p95_value']]

if 'trace_top50' not in globals() or trace_top50 is None or trace_top50.shape[0]==0:
    grp_t = trace_df.groupby(['cmdb_id','trace_name'], as_index=False).agg(
        count_of_points = ('value','count'),
        p95_value = ('value', lambda s: float(s.quantile(0.95)) if s.notna().any() else float('nan'))
    )
    trace_top50 = grp_t.sort_values('count_of_points', ascending=False).head(50)[['cmdb_id','trace_name','count_of_points','p95_value']]

# Define incident window in Asia/Shanghai, convert to UTC epoch seconds
start_local = tz.localize(pd.Timestamp('2020-05-27 00:30:00'))
end_local   = tz.localize(pd.Timestamp('2020-05-27 01:00:00'))
start_utc = start_local.astimezone(UTC)
end_utc   = end_local.astimezone(UTC)
start_epoch = int(start_utc.timestamp())
end_epoch   = int(end_utc.timestamp())

# 1) Filter metric.csv to the incident window and report total number of metric rows within window
metric_window = metric_df[(metric_df['timestamp'] >= start_epoch) & (metric_df['timestamp'] <= end_epoch)].copy()
metric_window_count = metric_window.shape[0]

# 2) For each (cmdb_id,kpi_name) present in metric_top50 compute anomalies using p95_value from metric_top50
# Merge p95 thresholds
mt = metric_top50[['cmdb_id','kpi_name','p95_value']].copy()
# Join metric_window with thresholds
mw = metric_window.merge(mt, on=['cmdb_id','kpi_name'], how='inner', suffixes=('','_p95'))
if mw.shape[0] == 0:
    metric_anomalies_out = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','max_value_in_incident','p95_value'])
else:
    # Determine anomalies where value >= p95_value
    mw['is_anom'] = mw['value'] >= mw['p95_value']
    grouped = mw.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
        anomaly_count = ('is_anom','sum'),
        earliest_anomaly_timestamp = ('timestamp', lambda s: int(s[mw.loc[s.index,'is_anom']].min()) if (mw.loc[s.index,'is_anom']).any() else pd.NA),
        max_value_in_incident = ('value','max'),
        p95_value = ('p95_value','first')
    )
    # Keep only anomaly_count>0
    metric_anomalies_out = grouped[grouped['anomaly_count']>0].copy()
    # Ensure earliest_anomaly_timestamp is integer (no NaN)
    metric_anomalies_out['earliest_anomaly_timestamp'] = metric_anomalies_out['earliest_anomaly_timestamp'].apply(lambda x: int(x) if pd.notna(x) else pd.NA)
    metric_anomalies_out = metric_anomalies_out.sort_values('anomaly_count', ascending=False).head(20).reset_index(drop=True)

# 3) Filter trace.csv to same incident window and report total trace rows within the window
trace_window = trace_df[(trace_df['timestamp'] >= start_epoch) & (trace_df['timestamp'] <= end_epoch)].copy()
trace_window_count = trace_window.shape[0]

# 4) For each (cmdb_id,trace_name) present in trace_top50 compute anomalies using p95_value from trace_top50
tt = trace_top50[['cmdb_id','trace_name','p95_value']].copy()
tw = trace_window.merge(tt, on=['cmdb_id','trace_name'], how='inner', suffixes=('','_p95'))
if tw.shape[0] == 0:
    trace_anomalies_out = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp','max_value_in_incident','p95_value'])
else:
    tw['is_anom'] = tw['value'] >= tw['p95_value']
    grouped_t = tw.groupby(['cmdb_id','trace_name'], as_index=False).agg(
        anomaly_count = ('is_anom','sum'),
        earliest_anomaly_timestamp = ('timestamp', lambda s: int(s[tw.loc[s.index,'is_anom']].min()) if (tw.loc[s.index,'is_anom']).any() else pd.NA),
        max_value_in_incident = ('value','max'),
        p95_value = ('p95_value','first')
    )
    trace_anomalies_out = grouped_t[grouped_t['anomaly_count']>0].copy()
    trace_anomalies_out['earliest_anomaly_timestamp'] = trace_anomalies_out['earliest_anomaly_timestamp'].apply(lambda x: int(x) if pd.notna(x) else pd.NA)
    trace_anomalies_out = trace_anomalies_out.sort_values('anomaly_count', ascending=False).head(20).reset_index(drop=True)

# Final compact outputs (single-row totals and up to 20 candidate rows)
metric_window_count, metric_anomalies_out, trace_window_count, trace_anomalies_out

```
Out[2]:
```
```python
# Summarize the previously computed incident-window anomaly analysis into a compact plain-English string.
# Reuse existing kernel variables: metric_window_count, metric_anomalies_out, trace_window_count, trace_anomalies_out

import pytz
import pandas as pd

tz = pytz.timezone('Asia/Shanghai')

lines = []
# Totals
try:
    lines.append(f"Total metric rows in incident window: {int(metric_window_count)}")
except Exception:
    lines.append("Total metric rows in incident window: <unknown>")

try:
    lines.append(f"Total trace rows in incident window: {int(trace_window_count)}")
except Exception:
    lines.append("Total trace rows in incident window: <unknown>")

lines.append("")  # spacer

# Metric anomalies summary (top 20 available, show top 10 for compactness)
if ('metric_anomalies_out' in globals()) and (hasattr(metric_anomalies_out, "shape")) and metric_anomalies_out.shape[0] > 0:
    lines.append(f"Metric anomalies (showing top {min(10, metric_anomalies_out.shape[0])} by anomaly_count):")
    for i, r in metric_anomalies_out.head(10).iterrows():
        et = r['earliest_anomaly_timestamp']
        et_s = pd.to_datetime(int(et), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z') if pd.notna(et) else "N/A"
        lines.append(f" - {r['cmdb_id']} | {r['kpi_name']}: count={int(r['anomaly_count'])}, earliest={et_s}, max_in_window={r['max_value_in_incident']}, p95={r['p95_value']}")
else:
    lines.append("Metric anomalies: <none found>")

lines.append("")  # spacer

# Trace anomalies summary
if ('trace_anomalies_out' in globals()) and (hasattr(trace_anomalies_out, "shape")) and trace_anomalies_out.shape[0] > 0:
    lines.append(f"Trace anomalies (showing top {min(10, trace_anomalies_out.shape[0])} by anomaly_count):")
    for i, r in trace_anomalies_out.head(10).iterrows():
        et = r['earliest_anomaly_timestamp']
        et_s = pd.to_datetime(int(et), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z') if pd.notna(et) else "N/A"
        lines.append(f" - {r['cmdb_id']} | {r['trace_name']}: count={int(r['anomaly_count'])}, earliest={et_s}, max_in_window={r['max_value_in_incident']}, p95={r['p95_value']}")
else:
    lines.append("Trace anomalies: <none found>")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(45621,     cmdb_id                  kpi_name  anomaly_count  earliest_anomaly_timestamp  max_value_in_incident   p95_value
0    os_007               Disk_rd_ios             31                  1590510600               0.000000    0.000000
1    os_020              Disk_io_util             31                  1590510600               0.000000    0.000000
2    os_012                Agent_ping             30                  1590510600               1.000000    1.000000
3    os_017                Agent_ping             30                  1590510600               1.000000    1.000000
4    os_016                   Page_pi             30                  1590510600               0.000000    0.000000
5   osb_001           app.succee_rate             30                  1590510600               1.000000    1.000000
6    db_007             MEM_real_util              7                  1590510720              80.300000   80.300000
7    os_012              Disk_io_util              6                  1590510900               1.000000    1.000000
8    os_012              CPU_idle_pct              4                  1590510780              99.588432   99.566221
9    os_017           Memory_used_pct              4                  1590512040               7.310000    7.290000
10   db_003              CPU_free_pct              4                  1590511380              99.329849   98.933695
11  osb_001            app.succee_num              4                  1590511680             557.000000  530.000000
12  osb_001                   app.num              4                  1590511680             557.000000  530.000000
13   os_017                Send_total              4                  1590511140               0.263568    0.259472
14   os_022  Outgoing_network_traffic              4                  1590511740               0.072546    0.068280
15   os_016                  ss_total              3                  1590511020             235.000000  223.100000
16   os_018           CPU_system_time              3                  1590511320               1.104499    1.075415
17   os_015  Incoming_network_traffic              3                  1590511320               0.003451    0.002354
18   os_012           CPU_iowait_time              3                  1590510900               0.411603    0.353976
19   os_008                Send_total              3                  1590510600               0.000687    0.000679, 6340,        cmdb_id                                     trace_name  anomaly_count  earliest_anomaly_timestamp  max_value_in_incident   p95_value
0   docker_001                      trace.self.csf.error_rate             30                  1590510600               0.000000    0.000000
1   docker_001     trace.from_os_022.remoteprocess.error_rate             30                  1590510600               0.000000    0.000000
2   docker_001                     trace.self.jdbc.error_rate             30                  1590510600               0.000000    0.000000
3   docker_001                trace.self.flyremote.error_rate             30                  1590510600               0.000000    0.000000
4   docker_002                    trace.self.local.error_rate             30                  1590510600               0.000000    0.000000
5   docker_002                     trace.self.jdbc.error_rate             30                  1590510600               0.000000    0.000000
6   docker_001   trace.to_docker_006.remoteprocess.error_rate             30                  1590510600               0.000000    0.000000
7   docker_002     trace.from_os_022.remoteprocess.error_rate             30                  1590510600               0.000000    0.000000
8   docker_001   trace.to_docker_005.remoteprocess.error_rate             30                  1590510600               0.000000    0.000000
9   docker_001                    trace.self.local.error_rate             30                  1590510600               0.000000    0.000000
10  docker_002                      trace.self.csf.error_rate             30                  1590510600               0.013699    0.000000
11  docker_002                trace.self.flyremote.error_rate             30                  1590510600               0.000000    0.000000
12  docker_001                   trace.self.jdbc.duration_p95              4                  1590512040               0.712000    0.711000
13  docker_001    trace.to_docker_005.remoteprocess.row_count              3                  1590510780             282.000000  275.000000
14  docker_001   trace.from_os_022.remoteprocess.duration_p95              2                  1590512100              49.919800   45.132622
15  docker_001                    trace.self.csf.duration_p95              2                  1590512040               4.251200    4.248335
16  docker_001                   trace.self.csf.duration_mean              2                  1590512100               2.767429    2.666777
17  docker_001  trace.from_os_022.remoteprocess.duration_mean              2                  1590512100              36.898583   35.730468
18  docker_001                  trace.self.local.duration_p95              2                  1590512160               1.416000    1.410965
19  docker_001                 trace.self.local.duration_mean              2                  1590512040               0.858121    0.808152)```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing kernel variables if present: metric_df, trace_df, metric_top50, trace_top50, metric_anomalies_out, trace_anomalies_out
# If any required variable missing, raise informative error by creating empty outputs.
required = ['metric_df','trace_df','metric_top50','trace_top50','metric_anomalies_out','trace_anomalies_out']
missing = [v for v in required if v not in globals()]
if missing:
    metric_runs_out = pd.DataFrame(columns=['cmdb_id','kpi_name','run_start_timestamp','run_end_timestamp','run_length_minutes','max_value_in_run','p95_value'])
    trace_runs_out  = pd.DataFrame(columns=['cmdb_id','trace_name','run_start_timestamp','run_end_timestamp','run_length_minutes','max_value_in_run','p95_value'])
    metric_runs_out, trace_runs_out
else:
    # Incident window in Asia/Shanghai -> convert to UTC epoch seconds
    start_local = tz.localize(pd.Timestamp('2020-05-27 00:30:00'))
    end_local   = tz.localize(pd.Timestamp('2020-05-27 01:00:00'))
    start_epoch = int(start_local.tz_convert(pytz.UTC).timestamp())
    end_epoch   = int(end_local.tz_convert(pytz.UTC).timestamp())

    # Ensure metric_top50 and trace_top50 contain p95_value columns
    if 'p95_value' not in metric_top50.columns:
        grp = metric_df.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
            p95_value=('value', lambda s: float(s.quantile(0.95)) if s.notna().any() else float('nan'))
        )
        metric_top50 = metric_top50.merge(grp, on=['cmdb_id','kpi_name'], how='left')
    if 'p95_value' not in trace_top50.columns:
        grp = trace_df.groupby(['cmdb_id','trace_name'], as_index=False).agg(
            p95_value=('value', lambda s: float(s.quantile(0.95)) if s.notna().any() else float('nan'))
        )
        trace_top50 = trace_top50.merge(grp, on=['cmdb_id','trace_name'], how='left')

    # Prepare windowed dataframes (reuse metric_window and trace_window if present)
    if 'metric_window' in globals():
        metric_window = metric_window
    else:
        metric_window = metric_df[(metric_df['timestamp'] >= start_epoch) & (metric_df['timestamp'] <= end_epoch)].copy()
    if 'trace_window' in globals():
        trace_window = trace_window
    else:
        trace_window = trace_df[(trace_df['timestamp'] >= start_epoch) & (trace_df['timestamp'] <= end_epoch)].copy()

    # Helper to compute runs for given windowed df and threshold table
    def compute_runs(window_df, top50_df, pairs_df, cmdb_col, name_col, value_col='value', p95_col='p95_value', top_pairs_limit=10):
        runs = []
        # select up to top_pairs_limit pairs from pairs_df sorted by anomaly_count desc
        if pairs_df is None or pairs_df.shape[0]==0:
            return pd.DataFrame(columns=[cmdb_col, name_col, 'run_start_timestamp','run_end_timestamp','run_length_minutes','max_value_in_run', p95_col])
        selected = pairs_df.sort_values('anomaly_count', ascending=False)[[cmdb_col, name_col]].drop_duplicates().head(top_pairs_limit)
        # build a dict for p95 lookup from top50_df
        p95_map = top50_df.set_index([cmdb_col, name_col])[p95_col].to_dict()
        for _, row in selected.iterrows():
            key = (row[cmdb_col], row[name_col])
            p95_val = p95_map.get(key, None)
            if p95_val is None or (isinstance(p95_val, float) and pd.isna(p95_val)):
                # skip if no p95 available
                continue
            sub = window_df[(window_df[cmdb_col]==row[cmdb_col]) & (window_df[name_col]==row[name_col])].copy()
            if sub.empty:
                continue
            sub = sub.sort_values('timestamp').reset_index(drop=True)
            # mark anomaly points
            sub['is_anom'] = sub[value_col] >= p95_val
            anom = sub[sub['is_anom']].copy()
            if anom.empty:
                continue
            # compute gaps between consecutive anomaly timestamps (60 sec expected)
            anom['ts_diff'] = anom['timestamp'].diff().fillna(9999)
            anom['new_run'] = (anom['ts_diff'] != 60).astype(int)
            anom['run_id'] = anom['new_run'].cumsum()
            grouped = anom.groupby('run_id', as_index=False).agg(
                run_start_timestamp = ('timestamp','min'),
                run_end_timestamp = ('timestamp','max'),
                run_length_minutes = ('timestamp','count'),
                max_value_in_run = (value_col,'max')
            )
            # attach identifiers and p95
            grouped[cmdb_col] = row[cmdb_col]
            grouped[name_col] = row[name_col]
            grouped[p95_col] = p95_val
            # keep runs length>=2
            grouped = grouped[grouped['run_length_minutes'] >= 2]
            for _, g in grouped.iterrows():
                runs.append({
                    cmdb_col: g[cmdb_col],
                    name_col: g[name_col],
                    'run_start_timestamp': int(g['run_start_timestamp']),
                    'run_end_timestamp': int(g['run_end_timestamp']),
                    'run_length_minutes': int(g['run_length_minutes']),
                    'max_value_in_run': float(g['max_value_in_run']),
                    p95_col: float(g[p95_col])
                })
        if not runs:
            return pd.DataFrame(columns=[cmdb_col, name_col, 'run_start_timestamp','run_end_timestamp','run_length_minutes','max_value_in_run', p95_col])
        runs_df = pd.DataFrame(runs)
        # sort by run_length desc then max_value_in_run desc
        runs_df = runs_df.sort_values(['run_length_minutes','max_value_in_run'], ascending=[False, False]).head(20).reset_index(drop=True)
        # ensure column order
        cols = [cmdb_col, name_col, 'run_start_timestamp','run_end_timestamp','run_length_minutes','max_value_in_run', p95_col]
        return runs_df[cols]

    # Compute metric runs
    metric_runs_out = compute_runs(metric_window, metric_top50, metric_anomalies_out, 'cmdb_id', 'kpi_name', value_col='value', p95_col='p95_value', top_pairs_limit=10)
    # Compute trace runs
    trace_runs_out  = compute_runs(trace_window, trace_top50, trace_anomalies_out, 'cmdb_id', 'trace_name', value_col='value', p95_col='p95_value', top_pairs_limit=10)

    # Return compact tables (up to 20 rows each)
    metric_runs_out, trace_runs_out

```
Out[3]:
```
```python
# Summarize top metric and trace anomaly runs (plain English) using kernel variables metric_runs_out and trace_runs_out
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

lines = []

# Metric runs summary
if 'metric_runs_out' not in globals():
    lines.append("Metric runs: <variable metric_runs_out not found in kernel>")
else:
    m = metric_runs_out
    if m is None or (hasattr(m, "shape") and m.shape[0] == 0):
        lines.append("Metric runs (run_length>=2): <none found>")
    else:
        lines.append(f"Metric runs (showing up to 10 of {m.shape[0]} runs):")
        for i, r in m.head(10).iterrows():
            start_ts = int(r['run_start_timestamp'])
            end_ts = int(r['run_end_timestamp'])
            start_s = pd.to_datetime(start_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
            end_s   = pd.to_datetime(end_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
            lines.append(f" - {r['cmdb_id']} | {r['kpi_name']}: length={int(r['run_length_minutes'])} min, start={start_s}, end={end_s}, max={r['max_value_in_run']}, p95={r['p95_value']}")

# Spacer
lines.append("")

# Trace runs summary
if 'trace_runs_out' not in globals():
    lines.append("Trace runs: <variable trace_runs_out not found in kernel>")
else:
    t = trace_runs_out
    if t is None or (hasattr(t, "shape") and t.shape[0] == 0):
        lines.append("Trace runs (run_length>=2): <none found>")
    else:
        lines.append(f"Trace runs (showing up to 10 of {t.shape[0]} runs):")
        for i, r in t.head(10).iterrows():
            start_ts = int(r['run_start_timestamp'])
            end_ts = int(r['run_end_timestamp'])
            start_s = pd.to_datetime(start_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
            end_s   = pd.to_datetime(end_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
            lines.append(f" - {r['cmdb_id']} | {r['trace_name']}: length={int(r['run_length_minutes'])} min, start={start_s}, end={end_s}, max={r['max_value_in_run']}, p95={r['p95_value']}")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

None```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')
UTC = pytz.UTC

# Reuse existing dataframes if present, else load from CSV
from pathlib import Path
if 'metric_df' not in globals():
    if Path('metric.csv').exists():
        metric_df = pd.read_csv('metric.csv')
    else:
        raise FileNotFoundError("metric.csv not found")
if 'trace_df' not in globals():
    if Path('trace.csv').exists():
        trace_df = pd.read_csv('trace.csv')
    else:
        raise FileNotFoundError("trace.csv not found")

# Normalize types
metric_df['timestamp'] = metric_df['timestamp'].astype(int)
trace_df['timestamp'] = trace_df['timestamp'].astype(int)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Incident window in Asia/Shanghai -> convert to UTC epoch seconds (inclusive)
start_local = tz.localize(pd.Timestamp('2020-05-27 00:30:00'))
end_local   = tz.localize(pd.Timestamp('2020-05-27 01:00:00'))
start_epoch = int(start_local.astimezone(UTC).timestamp())
end_epoch   = int(end_local.astimezone(UTC).timestamp())

# Ensure metric_top50 and trace_top50 have global p95_value; if missing, compute p95 map from full series
if 'metric_top50' in globals() and hasattr(metric_top50, 'columns') and 'p95_value' in metric_top50.columns:
    metric_p95_map = metric_top50.set_index(['cmdb_id','kpi_name'])['p95_value'].to_dict()
else:
    # compute global p95 per (cmdb_id,kpi_name) from full metric_df
    metric_p95_map = metric_df.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.95).astype(float).to_dict()
    # also build metric_top50 for reuse (counts+ p95) limited to top 50 by count
    metric_top50 = (metric_df.groupby(['cmdb_id','kpi_name'], as_index=False)
                        .agg(count_of_points=('value','count'))
                        .merge(pd.DataFrame([(k[0],k[1],v) for k,v in metric_p95_map.items()],
                                            columns=['cmdb_id','kpi_name','p95_value']),
                               on=['cmdb_id','kpi_name'], how='left')
                        .sort_values('count_of_points', ascending=False).head(50).reset_index(drop=True))

if 'trace_top50' in globals() and hasattr(trace_top50, 'columns') and 'p95_value' in trace_top50.columns:
    trace_p95_map = trace_top50.set_index(['cmdb_id','trace_name'])['p95_value'].to_dict()
else:
    trace_p95_map = trace_df.groupby(['cmdb_id','trace_name'])['value'].quantile(0.95).astype(float).to_dict()
    trace_top50 = (trace_df.groupby(['cmdb_id','trace_name'], as_index=False)
                      .agg(count_of_points=('value','count'))
                      .merge(pd.DataFrame([(k[0],k[1],v) for k,v in trace_p95_map.items()],
                                          columns=['cmdb_id','trace_name','p95_value']),
                             on=['cmdb_id','trace_name'], how='left')
                      .sort_values('count_of_points', ascending=False).head(50).reset_index(drop=True))

# Use metric_anomalies_out and trace_anomalies_out as sources of top pairs; if missing or empty -> return empty DataFrames
metric_runs_out = pd.DataFrame(columns=['cmdb_id','kpi_name','run_start_timestamp','run_end_timestamp','run_length_minutes','max_value_in_run','p95_value'])
trace_runs_out  = pd.DataFrame(columns=['cmdb_id','trace_name','run_start_timestamp','run_end_timestamp','run_length_minutes','max_value_in_run','p95_value'])

# Helper to compute runs for a pair
def compute_pair_runs(df_window, key_cols, value_col, p95_val):
    # df_window expected sorted by timestamp
    if df_window.empty:
        return []
    sub = df_window.sort_values('timestamp').reset_index(drop=True)
    # mark anomaly points
    sub['is_anom'] = sub[value_col] >= p95_val
    anom = sub[sub['is_anom']].copy()
    if anom.empty:
        return []
    anom['ts_diff'] = anom['timestamp'].diff().fillna(9999)
    anom['new_run'] = (anom['ts_diff'] != 60).astype(int)
    anom['run_id'] = anom['new_run'].cumsum()
    grouped = anom.groupby('run_id', as_index=False).agg(
        run_start_timestamp=('timestamp','min'),
        run_end_timestamp=('timestamp','max'),
        run_length_minutes=('timestamp','count'),
        max_value_in_run=(value_col,'max')
    )
    # filter run_length >=2
    grouped = grouped[grouped['run_length_minutes'] >= 2]
    return grouped.to_dict('records')

# Build metric runs
if 'metric_anomalies_out' in globals() and hasattr(metric_anomalies_out, 'shape') and metric_anomalies_out.shape[0] > 0:
    top_metric_pairs = (metric_anomalies_out.sort_values('anomaly_count', ascending=False)
                        [['cmdb_id','kpi_name']].drop_duplicates().head(10))
    runs = []
    # pre-filter metric_df to incident window to reuse
    metric_window = metric_df[(metric_df['timestamp'] >= start_epoch) & (metric_df['timestamp'] <= end_epoch)].copy()
    for _, pr in top_metric_pairs.iterrows():
        cm = pr['cmdb_id']; kpi = pr['kpi_name']
        p95 = metric_p95_map.get((cm,kpi), None)
        if p95 is None or (isinstance(p95,float) and pd.isna(p95)):
            continue
        sub = metric_window[(metric_window['cmdb_id']==cm) & (metric_window['kpi_name']==kpi)].copy()
        pair_runs = compute_pair_runs(sub, ['cmdb_id','kpi_name'], 'value', p95)
        for r in pair_runs:
            runs.append({
                'cmdb_id': cm,
                'kpi_name': kpi,
                'run_start_timestamp': int(r['run_start_timestamp']),
                'run_end_timestamp': int(r['run_end_timestamp']),
                'run_length_minutes': int(r['run_length_minutes']),
                'max_value_in_run': float(r['max_value_in_run']),
                'p95_value': float(p95)
            })
    if runs:
        metric_runs_out = (pd.DataFrame(runs)
                           .sort_values(['run_length_minutes','max_value_in_run'], ascending=[False, False])
                           .head(20)
                           .reset_index(drop=True))
    else:
        # keep empty DataFrame as required
        metric_runs_out = pd.DataFrame(columns=['cmdb_id','kpi_name','run_start_timestamp','run_end_timestamp','run_length_minutes','max_value_in_run','p95_value'])
else:
    # explicit empty if no anomalies source
    metric_runs_out = pd.DataFrame(columns=['cmdb_id','kpi_name','run_start_timestamp','run_end_timestamp','run_length_minutes','max_value_in_run','p95_value'])

# Build trace runs
if 'trace_anomalies_out' in globals() and hasattr(trace_anomalies_out, 'shape') and trace_anomalies_out.shape[0] > 0:
    top_trace_pairs = (trace_anomalies_out.sort_values('anomaly_count', ascending=False)
                       [['cmdb_id','trace_name']].drop_duplicates().head(10))
    runs_t = []
    trace_window = trace_df[(trace_df['timestamp'] >= start_epoch) & (trace_df['timestamp'] <= end_epoch)].copy()
    for _, pr in top_trace_pairs.iterrows():
        cm = pr['cmdb_id']; tn = pr['trace_name']
        p95 = trace_p95_map.get((cm,tn), None)
        if p95 is None or (isinstance(p95,float) and pd.isna(p95)):
            continue
        sub = trace_window[(trace_window['cmdb_id']==cm) & (trace_window['trace_name']==tn)].copy()
        pair_runs = compute_pair_runs(sub, ['cmdb_id','trace_name'], 'value', p95)
        for r in pair_runs:
            runs_t.append({
                'cmdb_id': cm,
                'trace_name': tn,
                'run_start_timestamp': int(r['run_start_timestamp']),
                'run_end_timestamp': int(r['run_end_timestamp']),
                'run_length_minutes': int(r['run_length_minutes']),
                'max_value_in_run': float(r['max_value_in_run']),
                'p95_value': float(p95)
            })
    if runs_t:
        trace_runs_out = (pd.DataFrame(runs_t)
                          .sort_values(['run_length_minutes','max_value_in_run'], ascending=[False, False])
                          .head(20)
                          .reset_index(drop=True))
    else:
        trace_runs_out = pd.DataFrame(columns=['cmdb_id','trace_name','run_start_timestamp','run_end_timestamp','run_length_minutes','max_value_in_run','p95_value'])
else:
    trace_runs_out = pd.DataFrame(columns=['cmdb_id','trace_name','run_start_timestamp','run_end_timestamp','run_length_minutes','max_value_in_run','p95_value'])

# Final compact outputs: metric_runs_out and trace_runs_out
metric_runs_out, trace_runs_out

```
Out[4]:
```
Summary of detected consecutive anomaly runs within 2020-05-27 00:30:00 — 2020-05-27 01:00:00 (Asia/Shanghai):

Metric runs (metric_runs_out)
- Total runs returned: 13 (filtered to run_length_minutes >= 2, up to 20 returned).
- Top runs by length:
  - os_007 | Disk_rd_ios — 31 min (1590510600 -> 1590512400), max=0.0, p95=0.0
  - os_020 | Disk_io_util — 31 min (1590510600 -> 1590512400), max=0.0, p95=0.0
  - os_012 | Agent_ping — 30 min (1590510600 -> 1590512340), max=1.0, p95=1.0
  - os_017 | Agent_ping — 30 min (1590510600 -> 1590512340), max=1.0, p95=1.0
  - os_016 | Page_pi — 30 min (1590510600 -> 1590512340), max=0.0, p95=0.0
  - osb_001 | app.succee_rate — two runs: 24 min (1590510600 -> 1590511980) and 6 min (1590512100 -> 1590512400), max=1.0, p95=1.0
  - additional shorter runs (length 2–4 min) for Memory_used_pct, CPU_idle_pct, MEM_real_util, Disk_io_util, etc.
- Note: epoch 1590510600 = 2020-05-27 00:30:00+08 and 1590512400 = 2020-05-27 01:00:00+08. Many long runs span the full incident window (30–31 minutes).

Trace runs (trace_runs_out)
- Total runs returned: 10 (filtered to run_length_minutes >= 2, up to 20 returned).
- Most prominent runs (each 30 min covering the window 1590510600 -> 1590512340):
  - docker_001 | trace.self.csf.error_rate — 30 min, max=0.0, p95=0.0
  - docker_001 | trace.from_os_022.remoteprocess.error_rate — 30 min, max=0.0, p95=0.0
  - docker_001 | trace.self.jdbc.error_rate — 30 min, max=0.0, p95=0.0
  - docker_001 | trace.self.flyremote.error_rate — 30 min, max=0.0, p95=0.0
  - docker_002 | trace.self.local.error_rate — 30 min, max=0.0, p95=0.0
  - docker_002 | trace.self.jdbc.error_rate — 30 min, max=0.0, p95=0.0
  - others include remoteprocess/error_rate and some shorter duration duration/row_count runs.
- Many trace error_rate runs are constant at or equal to their p95 (p95=0.0, max=0.0), producing long runs across the window.

Tables returned (as kernel variables)
- metric_runs_out: columns = [cmdb_id, kpi_name, run_start_timestamp, run_end_timestamp, run_length_minutes, max_value_in_run, p95_value], 13 rows.
- trace_runs_out: columns = [cmdb_id, trace_name, run_start_timestamp, run_end_timestamp, run_length_minutes, max_value_in_run, p95_value], 10 rows.

If you want, I can (a) convert the epoch timestamps to readable Asia/Shanghai datetimes in the tables, or (b) show only runs that do not simply equal p95 (to focus on values strictly above threshold).

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id         kpi_name  run_start_timestamp  run_end_timestamp  run_length_minutes  max_value_in_run  p95_value
0    os_007      Disk_rd_ios           1590510600         1590512400                  31          0.000000   0.000000
1    os_020     Disk_io_util           1590510600         1590512400                  31          0.000000   0.000000
2    os_012       Agent_ping           1590510600         1590512340                  30          1.000000   1.000000
3    os_017       Agent_ping           1590510600         1590512340                  30          1.000000   1.000000
4    os_016          Page_pi           1590510600         1590512340                  30          0.000000   0.000000
5   osb_001  app.succee_rate           1590510600         1590511980                  24          1.000000   1.000000
6   osb_001  app.succee_rate           1590512100         1590512400                   6          1.000000   1.000000
7    os_017  Memory_used_pct           1590512040         1590512220                   4          7.310000   7.290000
8    os_012     CPU_idle_pct           1590511860         1590511920                   2         99.588432  99.566221
9    db_007    MEM_real_util           1590512040         1590512100                   2         80.300000  80.300000
10   db_007    MEM_real_util           1590512340         1590512400                   2         80.300000  80.300000
11   os_012     Disk_io_util           1590510900         1590510960                   2          1.000000   1.000000
12   os_012     Disk_io_util           1590511380         1590511440                   2          1.000000   1.000000,       cmdb_id                                    trace_name  run_start_timestamp  run_end_timestamp  run_length_minutes  max_value_in_run  p95_value
0  docker_001                     trace.self.csf.error_rate           1590510600         1590512340                  30               0.0        0.0
1  docker_001    trace.from_os_022.remoteprocess.error_rate           1590510600         1590512340                  30               0.0        0.0
2  docker_001                    trace.self.jdbc.error_rate           1590510600         1590512340                  30               0.0        0.0
3  docker_001               trace.self.flyremote.error_rate           1590510600         1590512340                  30               0.0        0.0
4  docker_002                   trace.self.local.error_rate           1590510600         1590512340                  30               0.0        0.0
5  docker_002                    trace.self.jdbc.error_rate           1590510600         1590512340                  30               0.0        0.0
6  docker_001  trace.to_docker_006.remoteprocess.error_rate           1590510600         1590512340                  30               0.0        0.0
7  docker_002    trace.from_os_022.remoteprocess.error_rate           1590510600         1590512340                  30               0.0        0.0
8  docker_001  trace.to_docker_005.remoteprocess.error_rate           1590510600         1590512340                  30               0.0        0.0
9  docker_001                   trace.self.local.error_rate           1590510600         1590512340                  30               0.0        0.0)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Prepare output variables; reuse existing kernel variables metric_runs_out and trace_runs_out if present
expected_metric_cols = ['cmdb_id','kpi_name','run_start_timestamp','run_end_timestamp','run_length_minutes','max_value_in_run','p95_value']
expected_trace_cols  = ['cmdb_id','trace_name','run_start_timestamp','run_end_timestamp','run_length_minutes','max_value_in_run','p95_value']

# Metric breaches
if 'metric_runs_out' in globals() and isinstance(metric_runs_out, pd.DataFrame) and metric_runs_out.shape[0] > 0:
    metric_breach_df = metric_runs_out.copy()
    # Ensure numeric
    metric_breach_df['max_value_in_run'] = pd.to_numeric(metric_breach_df['max_value_in_run'], errors='coerce')
    metric_breach_df['p95_value'] = pd.to_numeric(metric_breach_df['p95_value'], errors='coerce')
    metric_breach_df['breach_amount'] = metric_breach_df['max_value_in_run'] - metric_breach_df['p95_value']
    # breach_ratio: NaN if max_value_in_run == 0 or NaN
    metric_breach_df['breach_ratio'] = np.where(
        metric_breach_df['max_value_in_run'] == 0,
        np.nan,
        metric_breach_df['breach_amount'] / metric_breach_df['max_value_in_run']
    )
    # Sort by breach_ratio desc (NaNs go to bottom), limit to top 20, select columns
    metric_breach_df = metric_breach_df.sort_values('breach_ratio', ascending=False).head(20)
    metric_breach_df = metric_breach_df[['cmdb_id','kpi_name','run_start_timestamp','run_end_timestamp','run_length_minutes','max_value_in_run','p95_value','breach_amount','breach_ratio']].reset_index(drop=True)
    # Count runs with breach_ratio > 0.5 (exclude NaN)
    metric_breach_count_gt_0_5 = int((metric_breach_df['breach_ratio'] > 0.5).sum())
else:
    # Empty-table indication
    metric_breach_df = pd.DataFrame(columns=expected_metric_cols + ['breach_amount','breach_ratio'])
    metric_breach_count_gt_0_5 = 0

# Trace breaches
if 'trace_runs_out' in globals() and isinstance(trace_runs_out, pd.DataFrame) and trace_runs_out.shape[0] > 0:
    trace_breach_df = trace_runs_out.copy()
    # Ensure numeric
    trace_breach_df['max_value_in_run'] = pd.to_numeric(trace_breach_df['max_value_in_run'], errors='coerce')
    trace_breach_df['p95_value'] = pd.to_numeric(trace_breach_df['p95_value'], errors='coerce')
    trace_breach_df['breach_amount'] = trace_breach_df['max_value_in_run'] - trace_breach_df['p95_value']
    trace_breach_df['breach_ratio'] = np.where(
        trace_breach_df['max_value_in_run'] == 0,
        np.nan,
        trace_breach_df['breach_amount'] / trace_breach_df['max_value_in_run']
    )
    trace_breach_df = trace_breach_df.sort_values('breach_ratio', ascending=False).head(20)
    trace_breach_df = trace_breach_df[['cmdb_id','trace_name','run_start_timestamp','run_end_timestamp','run_length_minutes','max_value_in_run','p95_value','breach_amount','breach_ratio']].reset_index(drop=True)
    trace_breach_count_gt_0_5 = int((trace_breach_df['breach_ratio'] > 0.5).sum())
else:
    trace_breach_df = pd.DataFrame(columns=expected_trace_cols + ['breach_amount','breach_ratio'])
    trace_breach_count_gt_0_5 = 0

# Display compact outputs: two tables and two scalars
metric_breach_df, metric_breach_count_gt_0_5, trace_breach_df, trace_breach_count_gt_0_5

```
Out[5]:
```
Summary of breach metrics computed from metric_runs_out and trace_runs_out:

Metric runs (metric_breach_df)
- Computed breach_amount = max_value_in_run - p95_value and breach_ratio = breach_amount / max_value_in_run (NaN when max_value_in_run == 0).
- Top rows by breach_ratio (showing the leading cases):
  - os_017 | Memory_used_pct — run 1590512040→1590512220, length=4 min, max=7.31, p95=7.29, breach_amount=0.02, breach_ratio≈0.002736
  - os_012 | CPU_idle_pct — run 1590511860→1590511920, length=2 min, max=99.588432, p95=99.566221, breach_amount≈0.022211, breach_ratio≈0.000223
  - Several runs (Agent_ping, app.succee_rate, MEM_real_util, Disk_io_util) have breach_amount = 0 (max == p95).
  - Some runs have max_value_in_run == 0 which yields breach_ratio = NaN.
- Count of metric runs with breach_ratio > 0.5: 0.

Trace runs (trace_breach_df)
- Same breach metrics computed.
- All top trace runs have max_value_in_run == p95_value (many are 0.0), so breach_amount = 0 and breach_ratio is NaN for zero max.
- Count of trace runs with breach_ratio > 0.5: 0.

Notes:
- No metric or trace run shows a large relative breach (breach_ratio > 0.5).
- The largest relative breaches are tiny (fractions of a percent) for the two metric runs listed above.

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id         kpi_name  run_start_timestamp  run_end_timestamp  run_length_minutes  max_value_in_run  p95_value  breach_amount  breach_ratio
0    os_017  Memory_used_pct           1590512040         1590512220                   4          7.310000   7.290000       0.020000      0.002736
1    os_012     CPU_idle_pct           1590511860         1590511920                   2         99.588432  99.566221       0.022211      0.000223
2    os_017       Agent_ping           1590510600         1590512340                  30          1.000000   1.000000       0.000000      0.000000
3    os_012       Agent_ping           1590510600         1590512340                  30          1.000000   1.000000       0.000000      0.000000
4   osb_001  app.succee_rate           1590512100         1590512400                   6          1.000000   1.000000       0.000000      0.000000
5   osb_001  app.succee_rate           1590510600         1590511980                  24          1.000000   1.000000       0.000000      0.000000
6    db_007    MEM_real_util           1590512040         1590512100                   2         80.300000  80.300000       0.000000      0.000000
7    db_007    MEM_real_util           1590512340         1590512400                   2         80.300000  80.300000       0.000000      0.000000
8    os_012     Disk_io_util           1590510900         1590510960                   2          1.000000   1.000000       0.000000      0.000000
9    os_012     Disk_io_util           1590511380         1590511440                   2          1.000000   1.000000       0.000000      0.000000
10   os_007      Disk_rd_ios           1590510600         1590512400                  31          0.000000   0.000000       0.000000           NaN
11   os_020     Disk_io_util           1590510600         1590512400                  31          0.000000   0.000000       0.000000           NaN
12   os_016          Page_pi           1590510600         1590512340                  30          0.000000   0.000000       0.000000           NaN, 0,       cmdb_id                                    trace_name  run_start_timestamp  run_end_timestamp  run_length_minutes  max_value_in_run  p95_value  breach_amount  breach_ratio
0  docker_001                     trace.self.csf.error_rate           1590510600         1590512340                  30               0.0        0.0            0.0           NaN
1  docker_001    trace.from_os_022.remoteprocess.error_rate           1590510600         1590512340                  30               0.0        0.0            0.0           NaN
2  docker_001                    trace.self.jdbc.error_rate           1590510600         1590512340                  30               0.0        0.0            0.0           NaN
3  docker_001               trace.self.flyremote.error_rate           1590510600         1590512340                  30               0.0        0.0            0.0           NaN
4  docker_002                   trace.self.local.error_rate           1590510600         1590512340                  30               0.0        0.0            0.0           NaN
5  docker_002                    trace.self.jdbc.error_rate           1590510600         1590512340                  30               0.0        0.0            0.0           NaN
6  docker_001  trace.to_docker_006.remoteprocess.error_rate           1590510600         1590512340                  30               0.0        0.0            0.0           NaN
7  docker_002    trace.from_os_022.remoteprocess.error_rate           1590510600         1590512340                  30               0.0        0.0            0.0           NaN
8  docker_001  trace.to_docker_005.remoteprocess.error_rate           1590510600         1590512340                  30               0.0        0.0            0.0           NaN
9  docker_001                   trace.self.local.error_rate           1590510600         1590512340                  30               0.0        0.0            0.0           NaN, 0)```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz
from pathlib import Path

tz = pytz.timezone('Asia/Shanghai')
UTC = pytz.UTC

# Ensure metric_df and trace_df are loaded in kernel
if 'metric_df' not in globals():
    if Path('metric.csv').exists():
        metric_df = pd.read_csv('metric.csv')
    else:
        raise FileNotFoundError("metric.csv not found")
if 'trace_df' not in globals():
    if Path('trace.csv').exists():
        trace_df = pd.read_csv('trace.csv')
    else:
        raise FileNotFoundError("trace.csv not found")

# Normalize types
metric_df['timestamp'] = metric_df['timestamp'].astype(int)
trace_df['timestamp'] = trace_df['timestamp'].astype(int)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Incident window boundaries in Asia/Shanghai -> convert to UTC epoch seconds (inclusive)
start_local = tz.localize(pd.Timestamp('2020-05-27 00:30:00'))
end_local   = tz.localize(pd.Timestamp('2020-05-27 01:00:00'))
start_epoch = int(start_local.astimezone(UTC).timestamp())
end_epoch   = int(end_local.astimezone(UTC).timestamp())

# 1) Low-end metric scan: compute global P5 per (cmdb_id,kpi_name) using full metric_df
p5_series = metric_df.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.05)
p5_df = p5_series.reset_index().rename(columns={'value':'p5_value'})

# Filter metric_df to incident window (use full-series p5 before filtering)
metric_window = metric_df[(metric_df['timestamp'] >= start_epoch) & (metric_df['timestamp'] <= end_epoch)].copy()
if metric_window.shape[0] == 0:
    metric_lowend_df = pd.DataFrame(columns=['cmdb_id','kpi_name','count_below_p5','earliest_timestamp_below_p5','min_value_in_window','p5_value'])
else:
    # Merge p5 thresholds into windowed data
    mw = metric_window.merge(p5_df, on=['cmdb_id','kpi_name'], how='left')
    rows = []
    for (cm, kpi), g in mw.groupby(['cmdb_id','kpi_name']):
        if g['p5_value'].isna().all():
            continue
        p5 = float(g['p5_value'].iloc[0])
        mask = g['value'] <= p5
        cnt = int(mask.sum())
        if cnt > 0:
            earliest = int(g.loc[mask, 'timestamp'].min())
            min_val = float(g['value'].min()) if g['value'].notna().any() else float('nan')
            rows.append({
                'cmdb_id': cm,
                'kpi_name': kpi,
                'count_below_p5': cnt,
                'earliest_timestamp_below_p5': earliest,
                'min_value_in_window': min_val,
                'p5_value': p5
            })
    if rows:
        metric_lowend_df = (pd.DataFrame(rows)
                            .sort_values('count_below_p5', ascending=False)
                            .head(20)
                            .reset_index(drop=True))
    else:
        metric_lowend_df = pd.DataFrame(columns=['cmdb_id','kpi_name','count_below_p5','earliest_timestamp_below_p5','min_value_in_window','p5_value'])

# 2) Looser-trace high-end scan: compute global P90 per (cmdb_id,trace_name) using full trace_df
p90_series = trace_df.groupby(['cmdb_id','trace_name'])['value'].quantile(0.90)
p90_df = p90_series.reset_index().rename(columns={'value':'p90_value'})

# Filter trace_df to incident window (use full-series p90 before filtering)
trace_window = trace_df[(trace_df['timestamp'] >= start_epoch) & (trace_df['timestamp'] <= end_epoch)].copy()
if trace_window.shape[0] == 0:
    trace_highend_df = pd.DataFrame(columns=['cmdb_id','trace_name','count_above_p90','earliest_timestamp_above_p90','max_value_in_window','p90_value'])
else:
    tw = trace_window.merge(p90_df, on=['cmdb_id','trace_name'], how='left')
    rows_t = []
    for (cm, tn), g in tw.groupby(['cmdb_id','trace_name']):
        if g['p90_value'].isna().all():
            continue
        p90 = float(g['p90_value'].iloc[0])
        mask = g['value'] >= p90
        cnt = int(mask.sum())
        if cnt > 0:
            earliest = int(g.loc[mask, 'timestamp'].min())
            max_val = float(g['value'].max()) if g['value'].notna().any() else float('nan')
            rows_t.append({
                'cmdb_id': cm,
                'trace_name': tn,
                'count_above_p90': cnt,
                'earliest_timestamp_above_p90': earliest,
                'max_value_in_window': max_val,
                'p90_value': p90
            })
    if rows_t:
        trace_highend_df = (pd.DataFrame(rows_t)
                            .sort_values('count_above_p90', ascending=False)
                            .head(20)
                            .reset_index(drop=True))
    else:
        trace_highend_df = pd.DataFrame(columns=['cmdb_id','trace_name','count_above_p90','earliest_timestamp_above_p90','max_value_in_window','p90_value'])

# Final compact outputs
metric_lowend_df, trace_highend_df

```
Out[6]:
```
Metric low-end scan (global P5; window 2020-05-27 00:30 → 01:00 Asia/Shanghai)
- Returned up to 20 (cmdb_id, kpi_name) pairs where at least one point in the window was <= the global P5.
- Top candidates (by count_below_p5):
  1. os_020 | Disk_io_util — count_below_p5=31, earliest=1590510600 (2020-05-27 00:30:00+08), min_in_window=0.0, p5=0.0
  2. osb_001 | app.succee_rate — count_below_p5=31, earliest=1590510600, min=0.974, p5=1.0
  3. os_007 | Disk_rd_ios — count_below_p5=31, earliest=1590510600, min=0.0, p5=0.0
  4. os_015 | Disk_io_util — count_below_p5=30, earliest=1590510600, min=0.0, p5=0.0
  5. docker_006 | container_fgct — count_below_p5=30, earliest=1590510600, min=0.0, p5=0.0
  6. docker_006 | container_fgc — count_below_p5=30, earliest=1590510600, min=0.0, p5=0.0
  7. docker_008 | container_thread_used_pct — count_below_p5=30, earliest=1590510600, min=0.0, p5=0.0
  8. db_007 | SctRead_Per_Sec — count_below_p5=30, earliest=1590510600, min=0.0, p5=0.0
  9. db_007 | Row_Lock — count_below_p5=30, earliest=1590510600, min=0.0, p5=0.0
 10. os_007 | Agent_ping — count_below_p5=30, earliest=1590510600, min=1.0, p5=1.0
- Observation: many low-end hits correspond to p5 == 0 (and min values 0), meaning the metric values in-window are at historically very low levels. A few metrics (e.g., app.succee_rate) have min < p5 (0.974 < 1.0) across the window.

Trace looser high-end scan (global P90; same window)
- Returned up to 20 (cmdb_id, trace_name) pairs with points >= global P90 in-window.
- Top candidates (by count_above_p90):
  - Many Docker traces show count_above_p90 = 30 with earliest=1590510600 (2020-05-27 00:30:00+08). Examples:
    - docker_001 | trace.from_os_022.remoteprocess.error_rate — count=30, max_in_window=0.0, p90=0.0
    - docker_001 | trace.self.csf.error_rate — count=30, max_in_window=0.0, p90=0.0
    - docker_001 | trace.self.jdbc.error_rate — count=30, max_in_window=0.0, p90=0.0
    - docker_002 | trace.self.csf.error_rate — count=30, earliest=1590510600, max_in_window=0.013699, p90=0.0 (only non-zero max observed)
  - Most trace p90 values are 0.0 and the max values in-window are 0.0 as well (so the "high-end" hits reflect being at or above a zero threshold).
- Observation: aside from a small non-zero case (docker_002 trace.self.csf.error_rate max=0.013699), trace values meeting or exceeding P90 are effectively zero-valued in the window.

Overall takeaway
- The low-end metric scan highlights many metrics at or below their historical P5 during the incident window (many with p5=0 and min=0).
- The trace high-end scan shows many trace series at/above their historical P90, but most of those P90 thresholds are 0, so few traces show meaningful positive latency spikes (one small non-zero example noted).
- If you want, I can (a) convert the epoch timestamps to readable datetimes in the returned tables, or (b) filter the results to exclude trivial zero-threshold cases to focus on non-zero deviations.

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                   kpi_name  count_below_p5  earliest_timestamp_below_p5  min_value_in_window   p5_value
0       os_020               Disk_io_util              31                   1590510600             0.000000   0.000000
1      osb_001            app.succee_rate              31                   1590510600             0.974000   1.000000
2       os_007                Disk_rd_ios              31                   1590510600             0.000000   0.000000
3       os_015               Disk_io_util              30                   1590510600             0.000000   0.000000
4   docker_006             container_fgct              30                   1590510600             0.000000   0.000000
5   docker_006              container_fgc              30                   1590510600             0.000000   0.000000
6   docker_008  container_thread_used_pct              30                   1590510600             0.000000   0.000000
7       db_007            SctRead_Per_Sec              30                   1590510600             0.000000   0.000000
8       db_007                   Row_Lock              30                   1590510600             0.000000   0.000000
9       os_007                 Agent_ping              30                   1590510600             1.000000   1.000000
10      os_007                Disk_wr_ios              30                   1590510600             0.000000   0.000000
11      os_007               Disk_io_util              30                   1590510600             0.000000   0.000000
12      os_007                 Disk_await              30                   1590510600             0.000000   0.000000
13      os_007                  ICMP_ping              30                   1590510600             1.000000   1.000000
14      os_006                  ICMP_ping              30                   1590510600             1.000000   1.000000
15      os_006                Disk_wr_ios              30                   1590510600             0.000000   0.000000
16      os_006                 Disk_svctm              30                   1590510600             0.000000   0.000000
17      os_006                Disk_rd_kbs              30                   1590510600             0.000000   0.000000
18      os_006            Memory_used_pct              30                   1590510600            18.010000  18.020000
19      os_006      Processor_load_15_min              30                   1590510600             0.008333   0.008333,        cmdb_id                                    trace_name  count_above_p90  earliest_timestamp_above_p90  max_value_in_window  p90_value
0   docker_001    trace.from_os_022.remoteprocess.error_rate               30                    1590510600             0.000000        0.0
1   docker_001                     trace.self.csf.error_rate               30                    1590510600             0.000000        0.0
2   docker_001                    trace.self.jdbc.error_rate               30                    1590510600             0.000000        0.0
3   docker_001               trace.self.flyremote.error_rate               30                    1590510600             0.000000        0.0
4   docker_001  trace.to_docker_006.remoteprocess.error_rate               30                    1590510600             0.000000        0.0
5   docker_002    trace.from_os_022.remoteprocess.error_rate               30                    1590510600             0.000000        0.0
6   docker_001  trace.to_docker_005.remoteprocess.error_rate               30                    1590510600             0.000000        0.0
7   docker_001                   trace.self.local.error_rate               30                    1590510600             0.000000        0.0
8   docker_002                     trace.self.csf.error_rate               30                    1590510600             0.013699        0.0
9   docker_002               trace.self.flyremote.error_rate               30                    1590510600             0.000000        0.0
10  docker_002                   trace.self.local.error_rate               30                    1590510600             0.000000        0.0
11  docker_002                    trace.self.jdbc.error_rate               30                    1590510600             0.000000        0.0
12  docker_003               trace.self.flyremote.error_rate               30                    1590510600             0.000000        0.0
13  docker_003    trace.from_os_021.remoteprocess.error_rate               30                    1590510600             0.000000        0.0
14  docker_002  trace.to_docker_006.remoteprocess.error_rate               30                    1590510600             0.000000        0.0
15  docker_002  trace.to_docker_005.remoteprocess.error_rate               30                    1590510600             0.000000        0.0
16  docker_003                    trace.self.jdbc.error_rate               30                    1590510600             0.000000        0.0
17  docker_003                   trace.self.local.error_rate               30                    1590510600             0.000000        0.0
18  docker_003  trace.to_docker_008.remoteprocess.error_rate               30                    1590510600             0.000000        0.0
19  docker_003                     trace.self.csf.error_rate               30                    1590510600             0.000000        0.0)```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz
from pathlib import Path

tz = pytz.timezone('Asia/Shanghai')
UTC = pytz.UTC

# Reuse existing metric_df and trace_df if available, else load
if 'metric_df' not in globals():
    if Path('metric.csv').exists():
        metric_df = pd.read_csv('metric.csv')
    else:
        raise FileNotFoundError("metric.csv not found")
if 'trace_df' not in globals():
    if Path('trace.csv').exists():
        trace_df = pd.read_csv('trace.csv')
    else:
        raise FileNotFoundError("trace.csv not found")

# Normalize types
metric_df['timestamp'] = metric_df['timestamp'].astype(int)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
trace_df['timestamp'] = trace_df['timestamp'].astype(int)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Incident window in Asia/Shanghai -> convert to UTC epoch seconds (inclusive)
start_local = tz.localize(pd.Timestamp('2020-05-27 00:30:00'))
end_local   = tz.localize(pd.Timestamp('2020-05-27 01:00:00'))
start_epoch = int(start_local.astimezone(UTC).timestamp())
end_epoch   = int(end_local.astimezone(UTC).timestamp())

# 1) osb_001 | app.succee_rate (metric)
metric_pair = ('osb_001', 'app.succee_rate')
mask_pair = (metric_df['cmdb_id'] == metric_pair[0]) & (metric_df['kpi_name'] == metric_pair[1])
if not mask_pair.any():
    metric_osb_rows = pd.DataFrame()  # empty
    count_rows_below_p5 = 0
    earliest_timestamp_below_p5 = pd.NA
    metric_osb_missing = f"MISSING: metric series {metric_pair[0]} | {metric_pair[1]} not found"
else:
    # global P5 computed from full series
    p5_val = float(metric_df.loc[mask_pair, 'value'].quantile(0.05))
    # filter to incident window first (per rules)
    metric_window = metric_df[(metric_df['timestamp'] >= start_epoch) & (metric_df['timestamp'] <= end_epoch)]
    metric_pair_window = metric_window[mask_pair.loc[metric_window.index]]
    # rows where value <= p5
    metric_osb_rows = metric_pair_window[metric_pair_window['value'] <= p5_val][['timestamp','cmdb_id','kpi_name','value']].sort_values('timestamp').head(50).reset_index(drop=True)
    count_rows_below_p5 = int(metric_pair_window[metric_pair_window['value'] <= p5_val].shape[0])
    earliest_timestamp_below_p5 = int(metric_pair_window.loc[metric_pair_window['value'] <= p5_val, 'timestamp'].min()) if count_rows_below_p5>0 else pd.NA
    metric_osb_missing = None

# 2) docker_002 | trace.self.csf.error_rate (trace)
trace_pair = ('docker_002', 'trace.self.csf.error_rate')
mask_tpair = (trace_df['cmdb_id'] == trace_pair[0]) & (trace_df['trace_name'] == trace_pair[1])
if not mask_tpair.any():
    trace_docker002_rows = pd.DataFrame()
    count_nonzero_rows_in_window = 0
    earliest_timestamp_nonzero_in_window = pd.NA
    trace_docker002_missing = f"MISSING: trace series {trace_pair[0]} | {trace_pair[1]} not found"
else:
    # compute global P90 from full series
    p90_val = float(trace_df.loc[mask_tpair, 'value'].quantile(0.90))
    # filter to incident window
    trace_window = trace_df[(trace_df['timestamp'] >= start_epoch) & (trace_df['timestamp'] <= end_epoch)]
    trace_pair_window = trace_window[mask_tpair.loc[trace_window.index]]
    # rows where value > 0 (non-zero) OR value >= p90 (to include >=p90 even if zero)
    mask_rows = (trace_pair_window['value'] > 0) | (trace_pair_window['value'] >= p90_val)
    trace_docker002_rows = trace_pair_window[mask_rows][['timestamp','cmdb_id','trace_name','value']].sort_values('timestamp').head(50).reset_index(drop=True)
    # scalars based on non-zero rows only
    nonzero_mask = (trace_pair_window['value'] > 0)
    count_nonzero_rows_in_window = int(nonzero_mask.sum())
    earliest_timestamp_nonzero_in_window = int(trace_pair_window.loc[nonzero_mask, 'timestamp'].min()) if count_nonzero_rows_in_window>0 else pd.NA
    trace_docker002_missing = None

# 3) docker_001 durations
duration_trace_names = [
    'trace.self.csf.duration_mean',
    'trace.self.csf.duration_p95',
    'trace.self.local.duration_mean',
    'trace.self.local.duration_p95',
    'trace.self.jdbc.duration_p95'
]
dur_rows_list = []
dur_summary_rows = []
missing_duration_series = []
# pre-filter trace_window for efficiency
trace_window = trace_df[(trace_df['timestamp'] >= start_epoch) & (trace_df['timestamp'] <= end_epoch)].copy()

for tn in duration_trace_names:
    mask_d = (trace_df['cmdb_id'] == 'docker_001') & (trace_df['trace_name'] == tn)
    if not mask_d.any():
        missing_duration_series.append(f"docker_001 | {tn}")
        continue
    # compute global P90 on full series
    p90 = float(trace_df.loc[mask_d, 'value'].quantile(0.90))
    # filter window for this pair
    mask_window = (trace_window['cmdb_id']=='docker_001') & (trace_window['trace_name']==tn)
    pair_window = trace_window[mask_window].sort_values('timestamp')
    if pair_window.empty:
        # no rows in window => summary with zero count
        dur_summary_rows.append({
            'cmdb_id':'docker_001',
            'trace_name':tn,
            'count_at_or_above_p90':0,
            'earliest_timestamp_at_or_above_p90':pd.NA,
            'p90_value':p90
        })
        continue
    # rows >= p90
    mask_at = pair_window['value'] >= p90
    rows_at = pair_window[mask_at][['timestamp','cmdb_id','trace_name','value']]
    count_at = int(mask_at.sum())
    earliest_at = int(pair_window.loc[mask_at, 'timestamp'].min()) if count_at>0 else pd.NA
    dur_summary_rows.append({
        'cmdb_id':'docker_001',
        'trace_name':tn,
        'count_at_or_above_p90':count_at,
        'earliest_timestamp_at_or_above_p90':earliest_at,
        'p90_value':p90
    })
    if count_at > 0:
        dur_rows_list.append(rows_at)

# Concatenate duration rows, limit total to 100 rows
if dur_rows_list:
    durations_rows = pd.concat(dur_rows_list, ignore_index=True).sort_values(['trace_name','timestamp']).head(100).reset_index(drop=True)
else:
    durations_rows = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value'])

durations_summary = pd.DataFrame(dur_summary_rows) if dur_summary_rows else pd.DataFrame(columns=['cmdb_id','trace_name','count_at_or_above_p90','earliest_timestamp_at_or_above_p90','p90_value'])

# Prepare missing messages
missing_msgs = []
if metric_osb_missing:
    missing_msgs.append(metric_osb_missing)
if trace_docker002_missing:
    missing_msgs.append(trace_docker002_missing)
if missing_duration_series:
    missing_msgs.append("MISSING duration series: " + ", ".join(missing_duration_series))
missing_series_message = "; ".join(missing_msgs) if missing_msgs else None

# Final compact outputs (variables)
metric_osb_rows, count_rows_below_p5, earliest_timestamp_below_p5, trace_docker002_rows, count_nonzero_rows_in_window, earliest_timestamp_nonzero_in_window, durations_rows, durations_summary, missing_series_message

```
Out[7]:
```
```python
# Summarize extracted anomaly rows for requested candidates using existing kernel variables.
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

def fmt_ts(epoch):
    try:
        return pd.to_datetime(int(epoch), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
    except Exception:
        return "N/A"

lines = []

# 1) osb_001 | app.succee_rate
if 'metric_osb_rows' not in globals():
    lines.append("Metric osb_001 | app.succee_rate: <variable metric_osb_rows not found>")
else:
    if isinstance(metric_osb_rows, pd.DataFrame) and metric_osb_rows.shape[0] > 0:
        lines.append("Metric osb_001 | app.succee_rate:")
        lines.append(f" - Rows in window with value <= global P5: {int(count_rows_below_p5)}")
        lines.append(f" - Earliest timestamp (epoch): {int(earliest_timestamp_below_p5) if pd.notna(earliest_timestamp_below_p5) else 'N/A'}")
        lines.append(f" - Earliest timestamp (Asia/Shanghai): {fmt_ts(earliest_timestamp_below_p5) if pd.notna(earliest_timestamp_below_p5) else 'N/A'}")
        lines.append(f" - Returned up to {min(50, metric_osb_rows.shape[0])} rows (timestamp, cmdb_id, kpi_name, value).")
    else:
        lines.append("Metric osb_001 | app.succee_rate: no rows <= global P5 in the incident window.")

# Spacer
lines.append("")

# 2) docker_002 | trace.self.csf.error_rate
if 'trace_docker002_rows' not in globals():
    lines.append("Trace docker_002 | trace.self.csf.error_rate: <variable trace_docker002_rows not found>")
else:
    if isinstance(trace_docker002_rows, pd.DataFrame) and trace_docker002_rows.shape[0] > 0:
        lines.append("Trace docker_002 | trace.self.csf.error_rate:")
        lines.append(f" - Non-zero rows in window: {int(count_nonzero_rows_in_window)}")
        et = earliest_timestamp_nonzero_in_window
        lines.append(f" - Earliest non-zero timestamp (epoch): {int(et) if pd.notna(et) else 'N/A'}")
        lines.append(f" - Earliest non-zero timestamp (Asia/Shanghai): {fmt_ts(et) if pd.notna(et) else 'N/A'}")
        lines.append(f" - Returned up to {min(50, trace_docker002_rows.shape[0])} rows (timestamp, cmdb_id, trace_name, value).")
        lines.append(" - Note: many returned rows may be zero-valued; non-zero count shown above.")
    else:
        lines.append("Trace docker_002 | trace.self.csf.error_rate: no rows in window (or no non-zero/p90 hits).")

# Spacer
lines.append("")

# 3) docker_001 duration traces (selected)
if 'durations_summary' not in globals() or 'durations_rows' not in globals():
    lines.append("docker_001 durations: <required variables not found>")
else:
    lines.append("docker_001 duration traces (rows where value >= global P90):")
    if durations_summary.shape[0] == 0:
        lines.append(" - No duration series found or no hits >= P90 in window.")
    else:
        # For each trace summary row: show count and earliest (converted)
        for _, r in durations_summary.iterrows():
            tn = r.get('trace_name')
            cnt = int(r.get('count_at_or_above_p90', 0))
            et = r.get('earliest_timestamp_at_or_above_p90')
            p90v = r.get('p90_value')
            lines.append(f" - {tn}: count_at_or_above_P90={cnt}, earliest_epoch={int(et) if pd.notna(et) else 'N/A'}, earliest_ASIA_SH={fmt_ts(et) if pd.notna(et) else 'N/A'}, p90={p90v}")
        total_rows = durations_rows.shape[0] if isinstance(durations_rows, pd.DataFrame) else 0
        lines.append(f" - Returned up to {min(100, total_rows)} row-level records across these traces (timestamp, cmdb_id, trace_name, value).")

# Missing series messages if any
if 'missing_series_message' in globals() and missing_series_message:
    lines.append("")
    lines.append(f"Missing-series note: {missing_series_message}")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     timestamp  cmdb_id         kpi_name  value
0   1590510600  osb_001  app.succee_rate    1.0
1   1590510660  osb_001  app.succee_rate    1.0
2   1590510720  osb_001  app.succee_rate    1.0
3   1590510780  osb_001  app.succee_rate    1.0
4   1590510840  osb_001  app.succee_rate    1.0
..         ...      ...              ...    ...
26  1590512160  osb_001  app.succee_rate    1.0
27  1590512220  osb_001  app.succee_rate    1.0
28  1590512280  osb_001  app.succee_rate    1.0
29  1590512340  osb_001  app.succee_rate    1.0
30  1590512400  osb_001  app.succee_rate    1.0

[31 rows x 4 columns], 31, 1590510600,      timestamp     cmdb_id                 trace_name  value
0   1590510600  docker_002  trace.self.csf.error_rate    0.0
1   1590510660  docker_002  trace.self.csf.error_rate    0.0
2   1590510720  docker_002  trace.self.csf.error_rate    0.0
3   1590510780  docker_002  trace.self.csf.error_rate    0.0
4   1590510840  docker_002  trace.self.csf.error_rate    0.0
..         ...         ...                        ...    ...
25  1590512100  docker_002  trace.self.csf.error_rate    0.0
26  1590512160  docker_002  trace.self.csf.error_rate    0.0
27  1590512220  docker_002  trace.self.csf.error_rate    0.0
28  1590512280  docker_002  trace.self.csf.error_rate    0.0
29  1590512340  docker_002  trace.self.csf.error_rate    0.0

[30 rows x 4 columns], 1, 1590512040,      timestamp     cmdb_id                      trace_name     value
0   1590512040  docker_001    trace.self.csf.duration_mean  2.602764
1   1590512100  docker_001    trace.self.csf.duration_mean  2.719151
2   1590512220  docker_001    trace.self.csf.duration_mean  2.767429
3   1590512040  docker_001     trace.self.csf.duration_p95  4.250900
4   1590512100  docker_001     trace.self.csf.duration_p95  4.245200
5   1590512220  docker_001     trace.self.csf.duration_p95  4.251200
6   1590512040  docker_001    trace.self.jdbc.duration_p95  0.712000
7   1590512100  docker_001    trace.self.jdbc.duration_p95  0.711000
8   1590512160  docker_001    trace.self.jdbc.duration_p95  0.711000
9   1590512220  docker_001    trace.self.jdbc.duration_p95  0.711000
10  1590512040  docker_001  trace.self.local.duration_mean  0.858121
11  1590512160  docker_001  trace.self.local.duration_mean  0.810355
12  1590512220  docker_001  trace.self.local.duration_mean  0.805459
13  1590512100  docker_001   trace.self.local.duration_p95  1.409700
14  1590512160  docker_001   trace.self.local.duration_p95  1.412000
15  1590512220  docker_001   trace.self.local.duration_p95  1.416000,       cmdb_id                      trace_name  count_at_or_above_p90  earliest_timestamp_at_or_above_p90  p90_value
0  docker_001    trace.self.csf.duration_mean                      3                          1590512040   2.549293
1  docker_001     trace.self.csf.duration_p95                      3                          1590512040   4.239440
2  docker_001  trace.self.local.duration_mean                      3                          1590512040   0.801450
3  docker_001   trace.self.local.duration_p95                      3                          1590512100   1.407270
4  docker_001    trace.self.jdbc.duration_p95                      4                          1590512040   0.711000, None)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```